# EchoLens: Ultimate Image Captioning Pipeline
This notebook trains the EfficientNetB0 + Transformer Encoder-Decoder model. It features **Dynamic Path Finding** (so it never crashes on Kaggle paths), **Fixed Visualization**, and **TFLite Export** for edge-device compression.

## 1. Dynamic Path Finding & Configuration

In [ ]:
import os
import re
import pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
# tf.config.run_functions_eagerly(True)
import keras
from keras import layers
from keras.applications import efficientnet
from keras.layers import TextVectorization

os.environ["KERAS_BACKEND"] = "tensorflow"
keras.utils.set_random_seed(111)

# DYNAMIC PATH FINDER
found_images_path = ""
found_captions_file = ""

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith(('.txt', '.csv')) and ('caption' in filename.lower() or 'result' in filename.lower()):
            found_captions_file = os.path.join(dirname, filename)
        if filename.endswith('.jpg') and not found_images_path:
            found_images_path = dirname

print(f"Dynamically Found Images Path: {found_images_path}")
print(f"Dynamically Found Captions File: {found_captions_file}")


# CONFIGURATION

config = {
    "IMAGES_PATH": found_images_path,
    "CAPTIONS_FILE": found_captions_file,
    "IMAGE_SIZE": (299, 299),
    "VOCAB_SIZE": 10000,
    "SEQ_LENGTH": 25,
    "EMBED_DIM": 512,
    "FF_DIM": 512,
    "BATCH_SIZE": 32,
    "EPOCHS": 15,
    "MODEL_PATH": "/kaggle/working/caption_model.keras",
    "VOCAB_PATH": "/kaggle/working/vectorization_layer_state.pkl"
}
AUTOTUNE = tf.data.AUTOTUNE

## 2. Data Loading and Preprocessing

In [ ]:
def load_captions_data(filename):
    with open(filename, encoding="utf8") as caption_file:
        caption_data = caption_file.readlines()
        caption_mapping = {}
        text_data = []
        images_to_skip = set()

        # Skip CSV Header if present
        if len(caption_data) > 0 and ("image" in caption_data[0].lower() or "caption" in caption_data[0].lower()):
            caption_data = caption_data[1:]

        for line in caption_data:
            line = line.replace('|', ',').rstrip("\n") # Handle weird delimiters just in case
            parts = line.strip().lower().split(",", 1)
            if len(parts) < 2: continue
            
            img_name, caption = parts
            img_path = os.path.join(config["IMAGES_PATH"], img_name.strip())

            tokens = caption.strip().split()
            if len(tokens) < 5 or len(tokens) > config["SEQ_LENGTH"]:
                images_to_skip.add(img_path)
                continue

            if img_path.endswith("jpg") and img_path not in images_to_skip:
                caption = "<start> " + caption.strip() + " <end>"
                text_data.append(caption)

                if img_path in caption_mapping:
                    caption_mapping[img_path].append(caption)
                else:
                    caption_mapping[img_path] = [caption]

        for img_path in images_to_skip:
            if img_path in caption_mapping:
                del caption_mapping[img_path]

        return caption_mapping, text_data

def train_val_split(caption_data, train_size=0.8, shuffle=True):
    all_images = list(caption_data.keys())
    if shuffle:
        np.random.shuffle(all_images)
    train_size = int(len(caption_data) * train_size)
    return (
        {img_name: caption_data[img_name] for img_name in all_images[:train_size]},
        {img_name: caption_data[img_name] for img_name in all_images[train_size:]}
    )

captions_mapping, text_data = load_captions_data(config["CAPTIONS_FILE"])
train_data, valid_data = train_val_split(captions_mapping)
print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(valid_data)}")

In [ ]:
strip_chars = r"!\"#$%&'()*+,-./:;<=>?@[\]^_`{|}~".replace("<", "").replace(">", "")
def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")

vectorization = TextVectorization(
    max_tokens=config["VOCAB_SIZE"],
    output_mode="int",
    output_sequence_length=config["SEQ_LENGTH"],
    standardize=custom_standardization,
)
vectorization.adapt(text_data)

image_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomContrast(0.3),
])

def decode_and_resize(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, config["IMAGE_SIZE"])
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img

def process_input(img_path, captions):
    return decode_and_resize(img_path), vectorization(captions)

# def make_dataset(images, captions):
#     dataset = tf.data.Dataset.from_tensor_slices((images, captions))
#     dataset = dataset.shuffle(config["BATCH_SIZE"] * 8)
#     dataset = dataset.map(process_input, num_parallel_calls=AUTOTUNE)
#     dataset = dataset.batch(config["BATCH_SIZE"]).prefetch(AUTOTUNE)
#     return dataset

def make_dataset(images, captions):
    dataset = tf.data.Dataset.from_tensor_slices((images, captions))
    dataset = dataset.shuffle(config["BATCH_SIZE"] * 8)
    dataset = dataset.map(process_input, num_parallel_calls=AUTOTUNE)
    
    # ADD THIS LINE TO CACHE IMAGES IN RAM:
    dataset = dataset.cache() 
    
    dataset = dataset.batch(config["BATCH_SIZE"]).prefetch(AUTOTUNE)
    return dataset

train_dataset = make_dataset(list(train_data.keys()), list(train_data.values()))
valid_dataset = make_dataset(list(valid_data.keys()), list(valid_data.values()))

## 3. Model Architecture (EfficientNet + Transformer)

In [ ]:
def get_cnn_model():
    base_model = efficientnet.EfficientNetB0(input_shape=(*config["IMAGE_SIZE"], 3), include_top=False, weights="imagenet")
    base_model.trainable = False
    base_model_out = layers.Reshape((-1, base_model.output.shape[-1]))(base_model.output)
    return keras.models.Model(base_model.input, base_model_out)

@keras.utils.register_keras_serializable()
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.0)
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.dense_1 = layers.Dense(embed_dim, activation="relu")

    def call(self, inputs, training, mask=None):
        inputs = self.layernorm_1(inputs)
        inputs = self.dense_1(inputs)
        attention_output_1 = self.attention_1(query=inputs, value=inputs, key=inputs, training=training)
        return self.layernorm_2(inputs + attention_output_1)

@keras.utils.register_keras_serializable()
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.embed_scale = tf.math.sqrt(tf.cast(embed_dim, tf.float32))

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs) * self.embed_scale
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return tf.math.not_equal(inputs, 0)

@keras.utils.register_keras_serializable()
class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, ff_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.attention_2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.ffn_layer_1 = layers.Dense(ff_dim, activation="relu")
        self.ffn_layer_2 = layers.Dense(embed_dim)
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.embedding = PositionalEmbedding(embed_dim=config["EMBED_DIM"], sequence_length=config["SEQ_LENGTH"], vocab_size=config["VOCAB_SIZE"])
        self.out = layers.Dense(config["VOCAB_SIZE"], activation="softmax")
        self.dropout_1 = layers.Dropout(0.3)
        self.dropout_2 = layers.Dropout(0.5)
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, training, mask=None):
        inputs = self.embedding(inputs)
        causal_mask = self.get_causal_attention_mask(inputs)
        if mask is not None:
            padding_mask = tf.cast(mask[:, :, tf.newaxis], dtype=tf.int32)
            combined_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32)
            combined_mask = tf.minimum(combined_mask, causal_mask)
        attention_output_1 = self.attention_1(query=inputs, value=inputs, key=inputs, attention_mask=combined_mask, training=training)
        out_1 = self.layernorm_1(inputs + attention_output_1)
        attention_output_2 = self.attention_2(query=out_1, value=encoder_outputs, key=encoder_outputs, attention_mask=padding_mask, training=training)
        out_2 = self.layernorm_2(out_1 + attention_output_2)
        ffn_out = self.ffn_layer_1(out_2)
        ffn_out = self.dropout_1(ffn_out, training=training)
        ffn_out = self.ffn_layer_2(ffn_out)
        ffn_out = self.layernorm_3(ffn_out + out_2, training=training)
        ffn_out = self.dropout_2(ffn_out, training=training)
        return self.out(ffn_out)

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = tf.concat([tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], axis=0)
        return tf.tile(mask, mult)

@keras.utils.register_keras_serializable()
class ImageCaptioningModel(keras.Model):
    def __init__(self, cnn_model, encoder, decoder, num_captions_per_image=5, image_aug=None, **kwargs):
        super().__init__(**kwargs)
        self.cnn_model = cnn_model
        self.encoder = encoder
        self.decoder = decoder
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.acc_tracker = keras.metrics.Mean(name="accuracy")
        self.num_captions_per_image = num_captions_per_image
        self.image_aug = image_aug

    def calculate_loss(self, y_true, y_pred, mask):
        loss = self.loss(y_true, y_pred)
        mask = tf.cast(mask, dtype=loss.dtype)
        loss *= mask
        return tf.reduce_sum(loss) / tf.reduce_sum(mask)

    def calculate_accuracy(self, y_true, y_pred, mask):
        accuracy = tf.equal(y_true, tf.argmax(y_pred, axis=2))
        accuracy = tf.math.logical_and(mask, accuracy)
        accuracy = tf.cast(accuracy, dtype=tf.float32)
        mask = tf.cast(mask, dtype=tf.float32)
        return tf.reduce_sum(accuracy) / tf.reduce_sum(mask)

    def _compute_caption_loss_and_acc(self, img_embed, batch_seq, training=True):
        encoder_out = self.encoder(img_embed, training=training)
        batch_seq_inp = batch_seq[:, :-1]
        batch_seq_true = batch_seq[:, 1:]
        mask = tf.math.not_equal(batch_seq_true, 0)
        batch_seq_pred = self.decoder(batch_seq_inp, encoder_out, training=training, mask=mask)
        loss = self.calculate_loss(batch_seq_true, batch_seq_pred, mask)
        acc = self.calculate_accuracy(batch_seq_true, batch_seq_pred, mask)
        return loss, acc

    def train_step(self, batch_data):
        batch_img, batch_seq = batch_data
        batch_loss = 0
        batch_acc = 0
        if self.image_aug:
            batch_img = self.image_aug(batch_img)
        img_embed = self.cnn_model(batch_img)

        for i in range(self.num_captions_per_image):
            with tf.GradientTape() as tape:
                loss, acc = self._compute_caption_loss_and_acc(img_embed, batch_seq[:, i, :], training=True)
                batch_loss += loss
                batch_acc += acc
            train_vars = self.encoder.trainable_variables + self.decoder.trainable_variables
            grads = tape.gradient(loss, train_vars)
            self.optimizer.apply_gradients(zip(grads, train_vars))

        batch_acc /= float(self.num_captions_per_image)
        self.loss_tracker.update_state(batch_loss)
        self.acc_tracker.update_state(batch_acc)
        return {"loss": self.loss_tracker.result(), "acc": self.acc_tracker.result()}

    def test_step(self, batch_data):
        batch_img, batch_seq = batch_data
        batch_loss = 0
        batch_acc = 0
        img_embed = self.cnn_model(batch_img)

        for i in range(self.num_captions_per_image):
            loss, acc = self._compute_caption_loss_and_acc(img_embed, batch_seq[:, i, :], training=False)
            batch_loss += loss
            batch_acc += acc

        batch_acc /= float(self.num_captions_per_image)
        self.loss_tracker.update_state(batch_loss)
        self.acc_tracker.update_state(batch_acc)
        return {"loss": self.loss_tracker.result(), "acc": self.acc_tracker.result()}

    @property
    def metrics(self):
        return [self.loss_tracker, self.acc_tracker]

    def get_config(self):
        conf = super().get_config()
        conf.update({
            "cnn_model": keras.saving.serialize_keras_object(self.cnn_model),
            "encoder": keras.saving.serialize_keras_object(self.encoder),
            "decoder": keras.saving.serialize_keras_object(self.decoder),
            "num_captions_per_image": self.num_captions_per_image,
            "image_aug": keras.saving.serialize_keras_object(self.image_aug),
        })
        return conf

    @classmethod
    def from_config(cls, conf):
        cnn_model = keras.saving.deserialize_keras_object(conf.pop("cnn_model"))
        encoder = keras.saving.deserialize_keras_object(conf.pop("encoder"))
        decoder = keras.saving.deserialize_keras_object(conf.pop("decoder"))
        image_aug = keras.saving.deserialize_keras_object(conf.pop("image_aug"))
        num_captions_per_image = conf.pop("num_captions_per_image")
        return cls(cnn_model=cnn_model, encoder=encoder, decoder=decoder, num_captions_per_image=num_captions_per_image, image_aug=image_aug, **conf)

    def build(self, input_shape, seq_len=25):
        dummy_images = tf.zeros(input_shape)
        if self.image_aug:
            dummy_images = self.image_aug(dummy_images)
        img_embed = self.cnn_model(dummy_images)
        encoder_out = self.encoder(img_embed, training=False)
        dummy_caption = tf.zeros((input_shape[0], seq_len), dtype=tf.int32)
        dummy_mask = tf.ones_like(dummy_caption, dtype=tf.bool)
        _ = self.decoder(dummy_caption, encoder_out, training=False, mask=dummy_mask)
        super().build(input_shape)

## 4. Training & Callbacks

In [ ]:
@keras.utils.register_keras_serializable()
class LRSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, post_warmup_learning_rate, warmup_steps):
        super().__init__()
        self.post_warmup_learning_rate = post_warmup_learning_rate
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        global_step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        warmup_progress = global_step / warmup_steps
        warmup_learning_rate = self.post_warmup_learning_rate * warmup_progress
        return tf.cond(global_step < warmup_steps, lambda: warmup_learning_rate, lambda: self.post_warmup_learning_rate)

    def get_config(self):
        return {"post_warmup_learning_rate": float(self.post_warmup_learning_rate), "warmup_steps": int(self.warmup_steps)}

    @classmethod
    def from_config(cls, conf):
        return cls(**conf)

cnn_model = get_cnn_model()
encoder = TransformerEncoderBlock(embed_dim=config["EMBED_DIM"], dense_dim=config["FF_DIM"], num_heads=1)
decoder = TransformerDecoderBlock(embed_dim=config["EMBED_DIM"], ff_dim=config["FF_DIM"], num_heads=2)
caption_model = ImageCaptioningModel(cnn_model=cnn_model, encoder=encoder, decoder=decoder, image_aug=image_augmentation)

cross_entropy = keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction=None)

num_train_steps = len(train_dataset) * config["EPOCHS"]
num_warmup_steps = num_train_steps // 15
lr_schedule = LRSchedule(post_warmup_learning_rate=1e-4, warmup_steps=num_warmup_steps)

caption_model.compile(optimizer=keras.optimizers.Adam(lr_schedule), loss=cross_entropy)
caption_model.build(input_shape=(4, *config["IMAGE_SIZE"], 3))

# Callbacks to save best model and stop if it stops improving
callbacks = [
    keras.callbacks.ModelCheckpoint(config["MODEL_PATH"], save_best_only=True, save_weights_only=False, monitor="val_loss"),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
]

print("Starting Training...")
history = caption_model.fit(
    train_dataset, 
    epochs=config["EPOCHS"], 
    validation_data=valid_dataset,
    callbacks=callbacks
)

## 5. Evaluation and Saving

In [ ]:
# Save vocabulary explicitly
with open(config["VOCAB_PATH"], "wb") as f:
    pickle.dump({'config': vectorization.get_config(), 'vocab': vectorization.get_vocabulary()}, f)
print(f"Vocabulary saved to {config['VOCAB_PATH']}")

# Plotting the training curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['acc'], label='Train Accuracy')
plt.plot(history.history['val_acc'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()
plt.show()

## 6. Inference / Sanity Check
Let's test the trained model on a random validation image. The visualization bug has been fixed so the image displays clearly.

In [ ]:
vocab = vectorization.get_vocabulary()
index_lookup = dict(zip(range(len(vocab)), vocab))
max_decoded_sentence_length = config["SEQ_LENGTH"] - 1
valid_images = list(valid_data.keys())

def generate_caption(img_path):
    sample_img = decode_and_resize(img_path)
    
    # IMAGE VISUALIZATION
    img_to_show = sample_img.numpy().astype("uint8")
    plt.imshow(img_to_show)
    plt.axis('off')
    plt.show()

    img = tf.expand_dims(sample_img, 0)
    img = caption_model.cnn_model(img)
    encoded_img = caption_model.encoder(img, training=False)

    decoded_caption = "<start> "
    for i in range(max_decoded_sentence_length):
        tokenized_caption = vectorization([decoded_caption])[:, :-1]
        mask = tf.math.not_equal(tokenized_caption, 0)
        predictions = caption_model.decoder(tokenized_caption, encoded_img, training=False, mask=mask)
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = index_lookup.get(sampled_token_index, "<unk>")
        if sampled_token == "<end>":
            break
        decoded_caption += " " + sampled_token

    decoded_caption = decoded_caption.replace("<start> ", "").replace(" <end>", "").strip()
    print("Predicted Caption: ", decoded_caption)
    
    return decoded_caption

# Test on a random image
random_test_img = np.random.choice(valid_images)
generate_caption(random_test_img)

## 7. The "Best" Step: TFLite Compression for Edge Devices
The standard `.keras` model is >200MB because it saves training states (optimizer weights). For real-time execution on a portable device (EchoLens), we compress it into two lightweight `TFLite` files (Encoder and Decoder), shrinking it down to roughly ~45MB total.

In [ ]:
import tensorflow as tf
from keras import layers

print("Isolating sub-models for TFLite conversion...")

# 1. Isolate the Image Encoder
image_input = tf.keras.Input(shape=(299, 299, 3), name="image_input")
cnn_features = caption_model.cnn_model(image_input)
encoded_features = caption_model.encoder(cnn_features, training=False)
encoder_model = tf.keras.Model(inputs=image_input, outputs=encoded_features, name="tflite_encoder")

# 2. Isolate the Text Decoder
seq_input = tf.keras.Input(shape=(None,), dtype=tf.int32, name="sequence_input")
enc_output_input = tf.keras.Input(shape=(None, config["EMBED_DIM"]), dtype=tf.float32, name="encoded_image_input")

# Wrap mask generation in a Lambda layer to handle KerasTensors ---
mask = layers.Lambda(lambda x: tf.math.not_equal(x, 0))(seq_input)

# Pass the mask into the decoder
preds = caption_model.decoder(seq_input, enc_output_input, training=False, mask=mask)
decoder_model = tf.keras.Model(inputs=[seq_input, enc_output_input], outputs=preds, name="tflite_decoder")

# 3. Setup Converter with Dynamic Range Quantization
def convert_and_save(model, filename):
    print(f"Converting {filename}...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    # Enable 8-bit dynamic range quantization to shrink model size by ~4x
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS, 
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    tflite_model = converter.convert()
    
    with open(f"/kaggle/working/{filename}", 'wb') as f:
        f.write(tflite_model)
    print(f"Saved /kaggle/working/{filename} (Size: {len(tflite_model) / (1024 * 1024):.2f} MB)")

# 4. Execute Conversion
convert_and_save(encoder_model, "echolens_encoder_quantized.tflite")
convert_and_save(decoder_model, "echolens_decoder_quantized.tflite")
print("TFLite models are ready for download in the Output panel!")